# Step 3 — Present score of EV-charging patent portfolios

Companion to **step2.1.ipynb**, using the same observed sector families and cleaned
applicant names. Code and explanations are in English, as in the Step 2 guide.

**Purpose:** compare applicants through volume, citation influence, EP grant success
and geographic filing breadth, then test how the ranking changes with weights and
aggregation. This is a relative portfolio score, not a monetary valuation or a
forecast. It follows the basic dimensions of course Tutorial 40, adapted to the
worldwide Step 2 sample rather than the tutorial's EP-only coffee sample.

**This file contains code, not real PATSTAT results.** It has been checked offline
with a small PATSTAT-shaped fixture. Run the real queries yourself on TIP.
No change to Step 2.1, new sector search or new name-cleaning method is required.

## 1. Connect Step 2 to this notebook

**Recommended, separate notebooks:** put `step2.1.ipynb` and `step3.ipynb` in the
same TIP directory. Run Step 2 from top to bottom. At the end of Step 2, run this
small additional cell to save its input tables (this is a new cell, no replacement):

```python
from pathlib import Path
import json
step3_input = Path("outputs/step2_for_step3")
step3_input.mkdir(parents=True, exist_ok=True)
raw_sector.to_csv(step3_input / "raw_sector.csv", index=False)
family_applicants.to_csv(step3_input / "family_applicants.csv", index=False)
family_years.to_csv(step3_input / "family_years.csv", index=False)
step3_config = {
    "ENV": ENV, "LAST_COMPLETE_YEAR": LAST_COMPLETE_YEAR,
    "SECTOR_NAME": SECTOR_NAME, "NAME_METHOD": NAME_METHOD,
    "SECTOR_CPC_PREFIXES": list(SECTOR_CPC_PREFIXES),
    "SECTOR_KEYWORDS": list(SECTOR_KEYWORDS),
    "VERIFIED_ALIASES": VERIFIED_ALIASES,
}
(step3_input / "step2_config.json").write_text(
    json.dumps(step3_config, indent=2), encoding="utf-8")
```

Then open Step 3, set `INPUT_MODE = "csv"`, and use **Restart Kernel and Run All**.
The environment is restored from the exported configuration, so TEST and PROD are
not silently mixed. These CSVs contain Step 2 inputs, not Step 3 metrics.

**Alternative, one kernel:** paste the code cells of Step 3 after Step 2 or select
the same running kernel for both notebooks. Use `INPUT_MODE = "memory"`.
If they share a folder, another manual option is to run `%run ./step2.1.ipynb`
in a new cell before Step 3; this re-executes Step 2 and its PATSTAT queries.
Step 3 never automatically executes Step 2.

If Step 0/1 changes the sector boundary, rerun Step 2 and re-export the inputs.
Do not pass only the top-15 firms: portfolio and cohort denominators need the full
Step 2 sample. Historical applicant links are not verified current ownership.

In [ ]:
# STEP3_CONFIGURATION
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Jupyter provides display; plain Python falls back to printable tables.
if "display" not in globals():
    def display(*objects):
        for obj in objects:
            print(obj)

INPUT_MODE = "memory"          # "memory" or "csv"; see instructions above
INPUT_DIR = Path("outputs/step2_for_step3")
STEP3_OUT = Path("outputs/step3")
EXPORT_STEP3 = False

# Fill these with the actual source information before presenting results.
PATSTAT_EDITION = None
SOURCE_OBSERVATION_DATE = None  # source coverage/snapshot date, not today's date
CITATION_COMPLETE_YEAR = None   # last fully covered citation year; do not guess

SCORE_PERIOD = (2010, 2018)      # tutorial starting point; test nearby windows
FORWARD_WINDOW = 5
N_FIRMS = 15
MIN_FAMILIES = 10
MIN_EP_APPLICATIONS = 5
MIN_INFLUENCE_COVERAGE = 0.80
MIN_COHORT_FAMILIES = 5
BATCH_SIZE = 500
WEIGHTS = {"volume": 0.25, "influence": 0.25,
           "success": 0.25, "globalization": 0.25}
INDICATORS = list(WEIGHTS)
NAVY, TEAL, GREY = "#17354A", "#087F8C", "#718096"

assert INPUT_MODE in {"memory", "csv"}
assert SCORE_PERIOD[0] <= SCORE_PERIOD[1]
assert BATCH_SIZE > 0 and FORWARD_WINDOW > 0 and N_FIRMS > 0
assert MIN_FAMILIES >= 1 and MIN_EP_APPLICATIONS >= 1 and MIN_COHORT_FAMILIES >= 1
assert 0 < MIN_INFLUENCE_COVERAGE <= 1
assert np.isclose(sum(WEIGHTS.values()), 1) and min(WEIGHTS.values()) >= 0

## 2. Read and validate the exact Step 2 inputs

`raw_sector` supplies application–applicant links and the existing `clean_name`.
`family_applicants` supplies unique family–applicant pairs and fractional credits;
`family_years` supplies the same minimum observed priority year as Step 2.
Company status is observed in `psn_sector`; it does not prove corporate ownership.
No fuzzy matching or new merger rules are applied.

In [ ]:
if INPUT_MODE == "csv":
    config = json.loads((INPUT_DIR / "step2_config.json").read_text(encoding="utf-8"))
    ENV = config["ENV"]
    LAST_COMPLETE_YEAR = int(config["LAST_COMPLETE_YEAR"])
    SECTOR_NAME = config["SECTOR_NAME"]
    NAME_METHOD = config["NAME_METHOD"]
    SECTOR_CPC_PREFIXES = tuple(config["SECTOR_CPC_PREFIXES"])
    SECTOR_KEYWORDS = tuple(config["SECTOR_KEYWORDS"])
    VERIFIED_ALIASES = config["VERIFIED_ALIASES"]
    raw_sector = pd.read_csv(INPUT_DIR / "raw_sector.csv", keep_default_na=False)
    family_applicants = pd.read_csv(INPUT_DIR / "family_applicants.csv", keep_default_na=False)
    family_years = pd.read_csv(INPUT_DIR / "family_years.csv", keep_default_na=False)
    from epo.tipdata.patstat import PatstatClient
    patstat = PatstatClient(env=ENV)
else:
    required = ["raw_sector", "family_applicants", "family_years", "patstat",
                "ENV", "LAST_COMPLETE_YEAR", "NAME_METHOD", "SECTOR_NAME",
                "SECTOR_CPC_PREFIXES", "SECTOR_KEYWORDS", "VERIFIED_ALIASES"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise ValueError("Run Step 2 in this kernel, or use csv mode. Missing: " + ", ".join(missing))

# Work on copies: the Step 2 DataFrames remain unchanged.
s3_raw = raw_sector.copy()
s3_links = family_applicants.copy()
s3_years = family_years.copy()
for frame, columns in [(s3_raw, ["appln_id", "docdb_family_id"]),
                       (s3_links, ["docdb_family_id", "earliest_filing_year"]),
                       (s3_years, ["docdb_family_id", "earliest_filing_year"])]:
    for column in columns:
        number = pd.to_numeric(frame[column], errors="raise")
        if number.isna().any() or not np.equal(number, np.floor(number)).all():
            raise ValueError(f"Missing or non-integer values in {column}")
        frame[column] = number.astype("int64")
s3_links["fractional_credit"] = pd.to_numeric(s3_links.fractional_credit, errors="raise")
assert not s3_links.duplicated(["docdb_family_id", "clean_name"]).any()
assert not s3_years.docdb_family_id.duplicated().any()
assert s3_links.clean_name.notna().all() and s3_links.clean_name.ne("").all()
assert s3_links.docdb_family_id.gt(0).all() and s3_raw.appln_id.gt(0).all()
assert set(s3_links.docdb_family_id) == set(s3_years.docdb_family_id)
assert set(s3_links.docdb_family_id) == set(s3_raw.docdb_family_id)
assert np.allclose(s3_links.groupby("docdb_family_id").fractional_credit.sum(), 1)
check_years = s3_links.merge(s3_years, on="docdb_family_id", suffixes=("_link", "_family"))
assert check_years.earliest_filing_year_link.eq(check_years.earliest_filing_year_family).all()
assert set(map(tuple, s3_raw[["docdb_family_id", "clean_name"]].drop_duplicates().values)) == set(
    map(tuple, s3_links[["docdb_family_id", "clean_name"]].values))
if not (SCORE_PERIOD[1] <= LAST_COMPLETE_YEAR):
    raise ValueError("The scoring period extends beyond Step 2's last complete filing year.")
citation_coverage_assumed = CITATION_COMPLETE_YEAR is None
if citation_coverage_assumed:
    CITATION_COMPLETE_YEAR = LAST_COMPLETE_YEAR
    print("CAUTION: citation coverage temporarily uses Step 2's filing cutoff. Verify and set the real citation coverage year.")
assert 1900 <= int(CITATION_COMPLETE_YEAR) < 9999
print("Source:", SECTOR_NAME, ENV, "| priority window:", SCORE_PERIOD,
      "| citation coverage through:", CITATION_COMPLETE_YEAR)
if not PATSTAT_EDITION or not SOURCE_OBSERVATION_DATE:
    print("DRAFT: fill PATSTAT_EDITION and SOURCE_OBSERVATION_DATE before final reporting.")

## 3. Scoring population and volume

The priority window initially follows Tutorial 40 (2010–2018); it is a design
choice to review against EV-charging coverage. All Step 2 families in that window
form the denominator, including those with no company applicant. Each observed
company receives a full attribution for its family. Co-applicant volume shares
can therefore sum above 100%. Fractional volume uses the original Step 2 credit,
divided among **all** observed applicants, not just companies.

Candidates are the top `N_FIRMS` companies with at least `MIN_FAMILIES`. The complete
population still supplies the denominators. Alphabetical tie-breaking makes a
top-N boundary reproducible; it does not imply a substantive difference.

In [ ]:
score_years = s3_years[s3_years.earliest_filing_year.between(*SCORE_PERIOD)].copy()
FAMILY_IDS = sorted(score_years.docdb_family_id.unique().tolist())
if not FAMILY_IDS:
    raise ValueError("No Step 2 families in SCORE_PERIOD: check the period and environment.")
score_links = s3_links[s3_links.docdb_family_id.isin(FAMILY_IDS)].copy()
company_pairs = s3_raw.loc[
    s3_raw.psn_sector.fillna("").astype(str).str.upper().str.contains("COMPANY", regex=False),
    ["docdb_family_id", "clean_name"]].drop_duplicates()
company_links = score_links.merge(company_pairs, on=["docdb_family_id", "clean_name"],
                                  how="inner", validate="one_to_one")
volume_table = company_links.groupby("clean_name").agg(
    families=("docdb_family_id", "nunique"),
    fractional_families=("fractional_credit", "sum"))
volume_table["volume"] = volume_table.families / len(FAMILY_IDS)
volume_table["volume_fractional"] = volume_table.fractional_families / len(FAMILY_IDS)
ordered = volume_table.reset_index().sort_values(["families", "clean_name"], ascending=[False, True])
CANDIDATES = ordered.loc[ordered.families.ge(MIN_FAMILIES), "clean_name"].head(N_FIRMS).tolist()
display(volume_table.sort_values("families", ascending=False).head(N_FIRMS))
print(len(FAMILY_IDS), "sector families;", len(volume_table), "companies;",
      len(CANDIDATES), "candidate companies")

## 4. Three small SQL extracts, restricted to Step 2 families

The batch helper uses the same `patstat.sql_query(..., use_legacy_sql=False)`
pattern as Step 2's SQL diagnostics. Integer lists are validated before insertion.
Queries only read tables. Database errors propagate: a failed query never becomes
an empty table or zero citations.

1. **Family metadata:** earliest publication year and distinct filing offices,
   across PI family members. `WO`, translations (`appln_kind = 'T'`) and artificial
   application IDs ≥900000000 are excluded from the office count. EP counts as
   one office. This is filing breadth, not active protection or EP validation states.
2. **EP flags:** grant flag of EP applications, joined later to the applicant of
   that very application observed in Step 2. No grant is assigned to a different
   co-applicant merely because it shares a family. EP filing year is retained for
   a five-year minimum-age check.
3. **Incoming family citations:** TLS228 pairs and the earliest publication year
   of the citing family. Citing families may come from outside the sector.

These additional records enrich existing families; they do not redefine the
sector or infer additional applicant links. The database can return more family
members than Step 2's observed application list.

In [ ]:
def query_batches(ids, sql_template, columns):
    ids = sorted(set(int(x) for x in ids))
    if any(x <= 0 for x in ids):
        raise ValueError("Only positive integer IDs are allowed")
    tables = []
    for start in range(0, len(ids), BATCH_SIZE):
        ids_sql = ", ".join(str(x) for x in ids[start:start + BATCH_SIZE])
        result = pd.DataFrame(patstat.sql_query(
            sql_template.format(ids=ids_sql), use_legacy_sql=False))
        if result.empty:
            continue
        result.columns = result.columns.str.lower()
        if not set(columns).issubset(result.columns):
            raise ValueError("Unexpected SQL result columns: " + str(result.columns.tolist()))
        tables.append(result[columns])
    return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame(columns=columns)

FAMILY_SQL = """
SELECT a.docdb_family_id,
       MIN(CASE WHEN a.earliest_publn_year BETWEEN 1800 AND 9998
                THEN a.earliest_publn_year END) AS first_publn_year,
       COUNT(DISTINCT CASE WHEN a.appln_auth <> 'WO'
                  AND TRIM(a.appln_auth) <> '' AND a.appln_kind <> 'T'
                  THEN a.appln_auth END) AS offices
FROM tls201_appln a
WHERE a.docdb_family_id IN ({ids}) AND a.ipr_type = 'PI'
  AND a.appln_id < 900000000
GROUP BY a.docdb_family_id
"""
EP_SQL = """
SELECT a.appln_id, a.docdb_family_id, a.appln_filing_year, a.granted
FROM tls201_appln a
WHERE a.docdb_family_id IN ({ids}) AND a.ipr_type = 'PI'
  AND a.appln_auth = 'EP' AND a.appln_kind <> 'T'
  AND a.appln_id < 900000000
"""
CITATION_SQL = """
SELECT c.docdb_family_id AS citing_family,
       c.cited_docdb_family_id AS cited_family,
       MIN(CASE WHEN a.earliest_publn_year BETWEEN 1800 AND 9998
                THEN a.earliest_publn_year END) AS citing_year
FROM tls228_docdb_fam_citn c
JOIN tls201_appln a ON a.docdb_family_id = c.docdb_family_id
WHERE c.cited_docdb_family_id IN ({ids}) AND c.docdb_family_id > 0
  AND c.docdb_family_id <> c.cited_docdb_family_id
  AND a.appln_id < 900000000
GROUP BY c.docdb_family_id, c.cited_docdb_family_id
"""
family_extra = query_batches(FAMILY_IDS, FAMILY_SQL,
    ["docdb_family_id", "first_publn_year", "offices"])
ep_extra = query_batches(FAMILY_IDS, EP_SQL,
    ["appln_id", "docdb_family_id", "appln_filing_year", "granted"])
citations = query_batches(FAMILY_IDS, CITATION_SQL,
    ["citing_family", "cited_family", "citing_year"])
assert not family_extra.docdb_family_id.duplicated().any()
assert not ep_extra.appln_id.duplicated().any()
assert not citations.duplicated(["citing_family", "cited_family"]).any()
print("Extracts:", len(family_extra), "families;", len(ep_extra),
      "EP applications;", len(citations), "incoming citation pairs")

## 5. Family influence and geographic breadth

As in the basic course tutorial, count distinct citing families whose **first
publication year** falls between the cited family's first publication year and
five years later (inclusive). This is a year-based citation proxy, not the actual
date of a citation event. A citation added much later may belong to an earlier
citing family. Self-family links are excluded; same-company citations remain.

Only families with a fully observed five-year window receive a count. Complete
families with no observed TLS228 links receive zero; incomplete families remain
missing. This assumes the source has reliable citation coverage; TEST is not
suitable for final influence estimates.

Divide counts by the mean of **all mature sector families with the same priority
year**, including uncited families. This normalization is sector/year-specific,
not global field normalization; the EV sector contains different technical fields.
Cohorts below `MIN_COHORT_FAMILIES` or with mean zero remain unscorable.
No arbitrary zero replaces an undefined ratio. Offices = 0 is a known count of
qualifying offices; absent family metadata remains missing.

In [ ]:
family_metrics = score_years.merge(family_extra, on="docdb_family_id",
                                   how="left", validate="one_to_one")
for col in ["first_publn_year", "offices"]:
    family_metrics[col] = pd.to_numeric(family_metrics[col], errors="raise")
family_metrics["citation_window_complete"] = (
    family_metrics.first_publn_year.notna() &
    family_metrics.first_publn_year.add(FORWARD_WINDOW).le(CITATION_COMPLETE_YEAR))
if not citations.empty:
    citations["citing_year"] = pd.to_numeric(citations.citing_year, errors="raise")
    citations = citations.merge(
        family_metrics[["docdb_family_id", "first_publn_year"]],
        left_on="cited_family", right_on="docdb_family_id", validate="many_to_one")
    citations["lag"] = citations.citing_year - citations.first_publn_year
    valid = citations[citations.lag.between(0, FORWARD_WINDOW) &
                      citations.citing_year.le(CITATION_COMPLETE_YEAR)]
    citation_counts = valid.groupby("cited_family").citing_family.nunique()
else:
    citation_counts = pd.Series(dtype=float)
family_metrics["fwd5"] = family_metrics.docdb_family_id.map(citation_counts).fillna(0).where(
    family_metrics.citation_window_complete)
cohorts = family_metrics.groupby("earliest_filing_year").fwd5
family_metrics["cohort_size"] = cohorts.transform("count")
family_metrics["cohort_mean"] = cohorts.transform("mean")
family_metrics["influence_norm"] = (family_metrics.fwd5 / family_metrics.cohort_mean).where(
    family_metrics.cohort_size.ge(MIN_COHORT_FAMILIES) & family_metrics.cohort_mean.gt(0))
display(family_metrics.groupby("earliest_filing_year").agg(
    families=("docdb_family_id", "size"), mature=("fwd5", "count"),
    mean_fwd5=("fwd5", "mean")))

## 6. Company indicators, with the correct denominators

Influence = mean normalized citations of the company's eligible families.
Influence coverage = families with a defined normalized influence / its full
portfolio in the score window. Globalization = mean offices across its families;
all families must have office metadata before the company enters the score.

EP success = EP applications flagged `Y` / EP applications with known `Y` or `N`,
restricted to applications filed at least five years before
`CITATION_COMPLETE_YEAR` (a conservative common cutoff). The grant flags describe
the **current source snapshot**, not grants within exactly five years or grants
known at a historical date. `N` can include pending cases; this is not a final
acceptance probability. An unknown flag is excluded and reported in coverage.
At least `MIN_EP_APPLICATIONS` mature, known applications are required, with all
mature EP flags known. Companies without observed EP applicant links stay missing.
EP-only companies may differ systematically from applicants using other offices.

In [ ]:
linked_metrics = company_links.merge(family_metrics.drop(columns="earliest_filing_year"),
                                     on="docdb_family_id", validate="many_to_one")
firm_indicators = volume_table.join(linked_metrics.groupby("clean_name").agg(
    influence=("influence_norm", "mean"),
    influence_families=("influence_norm", "count"),
    globalization=("offices", "mean"),
    globalization_families=("offices", "count")))
firm_indicators["influence_coverage"] = firm_indicators.influence_families / firm_indicators.families
firm_indicators["globalization_coverage"] = firm_indicators.globalization_families / firm_indicators.families

# An applicant can receive a grant attribution only through its own EP application.
application_names = s3_raw[["appln_id", "docdb_family_id", "clean_name"]].drop_duplicates()
ep_links = application_names.merge(ep_extra, on=["appln_id", "docdb_family_id"],
                                   how="inner", validate="many_to_one")
ep_links["appln_filing_year"] = pd.to_numeric(ep_links.appln_filing_year, errors="raise")
ep_links = ep_links[ep_links.appln_filing_year.between(1800, CITATION_COMPLETE_YEAR - 5)].copy()
ep_links["granted"] = ep_links.granted.fillna("").astype(str).str.strip().str.upper()
ep_links["known"] = ep_links.granted.isin(["Y", "N"])
ep_links["is_granted"] = ep_links.granted.eq("Y")
ep_summary = ep_links.groupby("clean_name").agg(
    mature_ep_observed=("appln_id", "nunique"),
    ep_applications=("known", "sum"), granted_ep=("is_granted", "sum"))
firm_indicators = firm_indicators.join(ep_summary)
for col in ["mature_ep_observed", "ep_applications", "granted_ep"]:
    firm_indicators[col] = firm_indicators[col].fillna(0).astype(int)
firm_indicators["success"] = firm_indicators.granted_ep.div(
    firm_indicators.ep_applications.replace(0, np.nan))
firm_indicators["ep_flag_coverage"] = firm_indicators.ep_applications.div(
    firm_indicators.mature_ep_observed.replace(0, np.nan))
display(firm_indicators.reindex(CANDIDATES))

## 7. Eligibility audit — keep excluded applicants visible

The thresholds are transparent analytical choices, not official standards.
The audit covers all observed companies; only top-volume candidates passing every
gate enter the ranking. Missing dimensions are not filled with zero and weights
are not silently redistributed. If no company passes, inspect coverage and source
first; do not automatically lower thresholds to manufacture a winner.

In [ ]:
eligibility = pd.DataFrame(index=firm_indicators.index)
eligibility["top_volume_candidate"] = eligibility.index.isin(CANDIDATES)
eligibility["portfolio_gate"] = firm_indicators.families.ge(MIN_FAMILIES)
eligibility["influence_gate"] = firm_indicators.influence_coverage.ge(MIN_INFLUENCE_COVERAGE)
eligibility["ep_gate"] = (firm_indicators.ep_applications.ge(MIN_EP_APPLICATIONS) &
                           firm_indicators.ep_flag_coverage.eq(1))
eligibility["offices_gate"] = firm_indicators.globalization_coverage.eq(1)
eligibility["complete_indicators"] = firm_indicators[INDICATORS].notna().all(axis=1)
eligibility["eligible"] = eligibility.all(axis=1)
gate_columns = [c for c in eligibility if c != "eligible"]
eligibility["excluded_because"] = eligibility[gate_columns].apply(
    lambda row: "; ".join(row.index[~row]), axis=1)
eligible_firms = firm_indicators.loc[eligibility.eligible].copy()
display(eligibility)
print("Eligible:", len(eligible_firms), "of", len(CANDIDATES), "candidates")

## 8. Weighted score and alternative rank aggregation

Each raw indicator is divided by its mean among the **eligible companies**.
The weighted mean is multiplied by 100. A score of 100 is the mean of this
comparison set; it is not 100% quality. Every eligible company uses the same
four dimensions and weights. A zero-mean dimension prevents score construction.

Reciprocal Rank Fusion (RRF) uses `rank(method="average")` for ties:
`100 × sum(weight × (c+1)/(c+rank))`, with `c = number of eligible companies`.
It dampens extreme numerical advantages. RRF and mean scores have different
scales; compare ranks, not score differences. With only one eligible company the
score is mechanically 100 and no meaningful competitive comparison is possible.

In [ ]:
def combine_score(table, weights, method="mean"):
    columns = list(weights)
    w = pd.Series(weights, dtype=float)
    if not np.isclose(w.sum(), 1) or w.lt(0).any():
        raise ValueError("Weights must be non-negative and sum to one")
    if table.empty:
        return pd.Series(index=table.index, dtype=float)
    values = table[columns]
    if values.isna().any().any() or not np.isfinite(values.to_numpy(dtype=float)).all():
        raise ValueError("Missing or non-finite indicators: check the eligibility audit")
    if method == "mean":
        means = values.mean()
        if means.le(0).any():
            raise ValueError("An indicator has zero mean; the four-dimension score is undefined")
        return 100 * values.div(means).mul(w).sum(axis=1)
    if method == "rrf":
        positions = values.rank(ascending=False, method="average")
        c = len(table)
        return 100 * ((c + 1) / (c + positions)).mul(w).sum(axis=1)
    raise ValueError("Unknown aggregation method")

ranking = eligible_firms.copy()
ranking["score"] = combine_score(eligible_firms, WEIGHTS)
ranking["score_rrf"] = combine_score(eligible_firms, WEIGHTS, "rrf")
ranking["rank"] = ranking.score.rank(ascending=False, method="min")
ranking["rank_rrf"] = ranking.score_rrf.rank(ascending=False, method="min")
ranking["rank_change_rrf"] = ranking["rank"] - ranking.rank_rrf
ranking = ranking.sort_values("score", ascending=False)
display(ranking)
if len(ranking) < 2:
    print("Insufficient eligible companies for a meaningful comparative ranking.")

## 9. Robustness: weights, fractional volume and omitted dimensions

The same eligible population and normalization reference are used for the
following comparisons. `impact_first` and `size_first` are proposed scenarios,
not estimated optimal weights. Positive rank change means the applicant moves up
relative to the baseline. A large rank range signals sensitivity, not uncertainty
measured by a statistical confidence interval.

For additional manual checks, rerun with nearby `SCORE_PERIOD` values and portfolio
thresholds; export each run separately and report any population changes. Do not
claim this notebook has already performed that period sensitivity test.

In [ ]:
scenarios = {
    "equal_weights": WEIGHTS,
    "impact_first": {"volume": .15, "influence": .40, "success": .20, "globalization": .25},
    "size_first": {"volume": .40, "influence": .15, "success": .20, "globalization": .25},
}
robust_scores = pd.DataFrame(index=ranking.index)
for name, weights in scenarios.items():
    robust_scores[name] = combine_score(eligible_firms, weights)
fractional_view = eligible_firms.copy()
fractional_view["volume"] = fractional_view.volume_fractional
robust_scores["fractional_volume"] = combine_score(fractional_view, WEIGHTS)
robust_scores["rrf"] = ranking.score_rrf
for omitted in INDICATORS:
    remaining = {k: v for k, v in WEIGHTS.items() if k != omitted}
    total = sum(remaining.values())
    if total > 0:
        remaining = {k: v / total for k, v in remaining.items()}
        robust_scores["without_" + omitted] = combine_score(eligible_firms, remaining)
robust_ranks = robust_scores.rank(ascending=False, method="min")
robust_ranks["best_rank"] = robust_ranks.min(axis=1)
robust_ranks["worst_rank"] = robust_ranks.drop(columns="best_rank").max(axis=1)
robust_ranks["rank_range"] = robust_ranks.worst_rank - robust_ranks.best_rank
display(robust_ranks)

## 10. Presentation figures

Use the ranking chart together with the indicator profile and sensitivity table.
The heatmap is relative to the same eligible-company mean (1 = reference mean);
the color scale alone is not a statement of patent quality. Report the sector,
priority window, number of eligible companies, weights, source edition, coverage
year and counting rule on the slide or in its methods note.

In [ ]:
if len(ranking) >= 2:
    fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(ranking))))
    ranking.score.sort_values().plot.barh(ax=ax, color=TEAL)
    ax.axvline(100, color=GREY, linestyle="--", label="eligible-company mean")
    ax.set(xlabel="Relative present score", ylabel="Applicant",
           title=f"{SECTOR_NAME}: present portfolio score ({ENV})")
    ax.legend(frameon=False)
    fig.tight_layout()
    if EXPORT_STEP3:
        STEP3_OUT.mkdir(parents=True, exist_ok=True)
        fig.savefig(STEP3_OUT / "present_score.png", dpi=160, bbox_inches="tight")
    plt.show()

    profile = ranking[INDICATORS].div(eligible_firms[INDICATORS].mean())
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(profile))))
    im = ax.imshow(profile.to_numpy(), aspect="auto", cmap="YlGnBu", vmin=0)
    ax.set_xticks(range(len(INDICATORS)), INDICATORS)
    ax.set_yticks(range(len(profile)), profile.index)
    for row in range(len(profile)):
        for col in range(len(INDICATORS)):
            value = profile.iloc[row, col]
            text_color = "white" if im.norm(value) > 0.55 else "black"
            ax.text(col, row, f"{value:.2f}", ha="center", va="center", color=text_color)
    ax.set_title("Indicator profile: ratio to eligible-company mean")
    fig.colorbar(im, ax=ax, label="1 = comparison-set mean")
    fig.tight_layout()
    if EXPORT_STEP3:
        fig.savefig(STEP3_OUT / "indicator_profile.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Charts omitted: at least two eligible companies are needed.")

## 11. Optional exports and provenance

Set `EXPORT_STEP3 = True` to save final tables, query templates, source fingerprints
and metadata. Set `STEP3_OUT` to a folder named for the source environment and
scoring period; use a fresh folder when preserving different runs. Reusing the
same folder overwrites its exports. Exporting a draft does not
turn TEST counts into sector findings. Source hashes track table content, not
the database snapshot. `source_application_names.csv` is the audit bridge.

Step 2 SDG proposals are **not included in the valuation score**: they measure
possible relevance to an objective, not technical quality or realized impact.
You may discuss manually reviewed SDG associations alongside the score, keeping
their evidence and denominator separate. No SDG classification is inferred here.

In [ ]:
def frame_fingerprint(frame):
    # Deterministic for the current serialized table order and values.
    return hashlib.sha256(frame.to_csv(index=False).encode("utf-8")).hexdigest()

run_metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_notebook": "step2.1.ipynb", "environment": ENV,
    "patstat_edition": PATSTAT_EDITION,
    "source_observation_date": SOURCE_OBSERVATION_DATE,
    "draft": ENV != "PROD" or not PATSTAT_EDITION or not SOURCE_OBSERVATION_DATE or citation_coverage_assumed,
    "citation_coverage_assumed_from_step2": citation_coverage_assumed,
    "sector": SECTOR_NAME, "cpc_prefixes": list(SECTOR_CPC_PREFIXES),
    "keywords": list(SECTOR_KEYWORDS), "name_method": NAME_METHOD,
    "verified_aliases": VERIFIED_ALIASES,
    "score_period": list(SCORE_PERIOD), "citation_complete_year": int(CITATION_COMPLETE_YEAR),
    "forward_window": FORWARD_WINDOW, "citation_method": "TLS228 first-publication-year lag; corporate self citations included",
    "success_method": "current source grant flags on mature observed EP applicant links",
    "globalization_method": "distinct PI filing offices, excluding WO and kind T",
    "weights": WEIGHTS, "n_firms_limit": N_FIRMS,
    "min_families": MIN_FAMILIES, "min_ep_applications": MIN_EP_APPLICATIONS,
    "min_influence_coverage": MIN_INFLUENCE_COVERAGE, "min_cohort_families": MIN_COHORT_FAMILIES,
    "sector_families": len(FAMILY_IDS), "candidates": len(CANDIDATES), "eligible": len(ranking),
    "source_hashes": {"raw_sector": frame_fingerprint(s3_raw),
                      "family_applicants": frame_fingerprint(s3_links),
                      "family_years": frame_fingerprint(s3_years)},
    "query_templates": {"families": FAMILY_SQL, "ep": EP_SQL, "citations": CITATION_SQL},
}
if EXPORT_STEP3:
    STEP3_OUT.mkdir(parents=True, exist_ok=True)
    for name, table, use_index in [
        ("present_score", ranking, True), ("firm_indicators", firm_indicators, True),
        ("eligibility_audit", eligibility, True), ("robustness_ranks", robust_ranks, True),
        ("robustness_scores", robust_scores, True), ("family_metrics", family_metrics, False),
        ("source_application_names", application_names, False), ("ep_applicant_audit", ep_links, False)]:
        table.to_csv(STEP3_OUT / f"{name}.csv", index=use_index)
    (STEP3_OUT / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
    print("Saved:", STEP3_OUT.resolve())
else:
    print("Tables remain in memory. Set EXPORT_STEP3=True for CSV, JSON and PNG output.")

## 12. Manual TIP checks and interpretation

1. Start with Step 2 and Step 3 on **TEST**. Verify input counts, uniqueness checks,
   SQL output schemas and missingness; TEST is for debugging, not market conclusions.
2. Examine `family_metrics`: missing publication years, incomplete citation
   windows, small/zero cohorts and unknown offices. Check several family IDs
   manually using the same SQL queries with a one-ID list.
3. Examine `ep_links` for a company with co-applicants: each success attribution
   must follow the applicant's own observed EP application. Inspect the age of
   applications and distinguish pending flags from final rejections.
4. Set the actual edition, observation date and citation coverage year. Verify
   that score period, citation window and EP maturity are defensible. The initial
   common cutoff is conservative; Step 2's filing cutoff is not proof of citation
   completeness. No historical backtest is performed by filtering a current source.
5. For final results, rerun **Step 2 on PROD**, regenerate its inputs, then run
   Step 3 on that same source. A PROD connection to TEST-selected IDs is not a
   full sector analysis. Inspect `eligibility` before comparing scores.
6. Explain score differences with the four underlying indicators, coverage and
   portfolio size. A high score does not establish patent validity, enforceability,
   revenue, acquisition suitability or current ownership.
7. Compare the weight and RRF scenarios. If leaders change, present multiple
   plausible rankings and explain the tradeoff. Period sensitivity requires
   separate reruns. Keep small samples and excluded firms visible in the appendix.

**Suggested Step 3 slides:** (1) scoring method and coverage; (2) present-score
ranking plus four-indicator profile; (3) sensitivity and implications. Place the
full audit and formulas in the appendix. A later strategic positioning analysis
can reuse `present_score.csv`; future potential requires additional evidence.

### Course sources and attribution

- Tutorial 40, *Present value score*: four dimensions, mean normalization, RRF,
  portfolio gates and weight sensitivity. Local course copy:
  `tmp/step2_step3_sources/tutorial_4__40_present_value_score.ipynb`.
- Tutorial 30, *Patent citations*: PATSTAT family/publication citation structures.
- Project kickoff, Step 3: present portfolio score; the later project steps are
  separate analyses, not outputs invented by this notebook.
- Step 2.1 provides the actual sector query, observed priority years, names and
  full/fractional applicant links. This companion adds no group-merger inference.

Course repository: https://github.com/graziasveva/innovation-management-datascience26
(course tutorials adapted from EPO TIP materials; this adaptation is not an
official EPO product). The score is inspired by the course, not a reproduction
of Clarivate's proprietary ranking.

### Implementation record

Prepared against Step 2.1 SHA-256 `54996be7f2be4217be5879ba31dbe775737c0816f3d9a9f8becc2120ede3980d`.
The original Step 2.1 file is unchanged. All outputs in this delivered notebook
are cleared. Offline verification uses a separate invented fixture, never labelled
as real PATSTAT evidence. The real TIP client and live query runtime still require
the manual checks above.